In [57]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 100)

# Read the Excel file into a DataFrame
df = pd.read_excel("C:/Users/user/Desktop/Python_Data_Project/risk_credit_card_analysis/data/raw/default of credit card clients.xls", header=1)

In [ ]:
# Tidy the column names (lowercase, no spaces, set proper namings for pay,bill,paid columns)
df.columns = df.columns.str.lower().str.replace(" ", "_")
df.columns = ['id', 'limit_bal', 'sex', 'education', 'marital_status', 'age',
            'pay_sep', 'pay_aug', 'pay_jul', 'pay_jun', 'pay_may', 'pay_apr',
            'bill_sep', 'bill_aug', 'bill_jul', 'bill_jun', 'bill_may', 'bill_apr',
            'paid_sep', 'paid_aug', 'paid_jul', 'paid_jun', 'paid_may', 'paid_apr',
            'is_default']

# Change the types of the columns for efficient memory allocation
df['sex'] = df['sex'].map({1: 'Male', 2: 'Female'})
df['sex'] = df['sex'].astype('category')

df['education'] = df['education'].map(
    {1:'Graduate School',
     2:'University',
     3:'High School',
     4:'Others'}
)
df['education'] = df['education'].fillna('Unknown')
df['education'] = df['education'].astype('category')

df['marital_status'] = df['marital_status'].map(
    {1:'Married',
     2:'Single',
     3:'Others'}
)
df['marital_status'] = df['marital_status'].fillna('Unknown')
df['marital_status'] = df['marital_status'].astype('category')

# Create column age_band to group the age column
df['age_band'] = pd.cut(
    df['age'],
    bins=[-np.inf, 25, 35, 45, 55, np.inf],
    labels=['<25', '25-34', '35-44', '45-54', '55+']
)

## Default rate segments

In [178]:
# Create benchmark for defaults
default_benchmark = df.agg(
    a = ('is_default', 'sum'),
    b = ('is_default', 'mean'),
    c = ('limit_bal', 'mean'),
)

# Default rate by segment
def defaults(colName):
    df_default = df.groupby(colName,
                            observed=True
    ).agg(
        count = ('is_default', 'size'),
        total = ('is_default', 'sum'),
        default_rate_pct = ('is_default', 'mean')
    ).round(4)
    df_default['default_rate_pct'] = df_default['default_rate_pct']*100
    df_default['share_defaults'] = df_default['total']/df['is_default'].sum()
    df_default = df_default.reset_index().rename(columns={colName:'dimension'})
    df_default = df_default.sort_values(by=['dimension', 'default_rate_pct'], ascending=[True,False])
    return df_default

default_education = defaults('education')
default_marital_status = defaults('marital_status')
default_age_band = defaults('age_band')
default_sex = defaults('sex')

default_segments = pd.concat([default_education, default_marital_status,
           default_age_band, default_sex], ignore_index=False)

default_segments

,dimension,count,total,default_rate_pct,share_defaults
0,Graduate School,10585,2036,19.23,0.306811
1,High School,4917,1237,25.16,0.186407
2,Others,123,7,5.69,0.001055
3,University,14030,3330,23.73,0.501808
4,Unknown,345,26,7.54,0.003918
0,Married,13659,3206,23.47,0.483122
1,Others,323,84,26.01,0.012658
2,Single,15964,3341,20.93,0.503466
3,Unknown,54,5,9.26,0.000753
0,<25,3871,1032,26.66,0.155515
